# RAIS 2022

Municipal formal-employment book for reference year 2022.
Variable definitions are in `dictionary.csv`.
The saved table includes the municipality key, year and source names.


## Imports

In [ ]:
import re
import numpy as np
import pandas as pd

from pathlib import Path
from unidecode import unidecode


## Configuração

In [ ]:
def resolve_repo_root() -> Path:
    """Locate the project root that contains the local data directory."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"

RAW_RAIS_DIR = BASE_DIR / "raw" / "rais"
TRUSTED_RAIS_DIR = BASE_DIR / "trusted" / "rais"
TRUSTED_IBGE_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_RAIS_DIR, TRUSTED_RAIS_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Diretório

In [ ]:
path_rais = RAW_RAIS_DIR / "rais_2022.csv"


## Nome de Colunas

In [ ]:
def normalizar_nome_coluna(col: str) -> str:
    """
    Normaliza nome de coluna:
    - remove acentos;
    - deixa minúsculo;
    - troca caracteres especiais por underscore.
    """
    col = str(col).strip()
    col = unidecode(col)
    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalizar_nome_coluna(col) for col in df.columns]
    return df


def parse_numero(s: pd.Series) -> pd.Series:
    """
    Parse numeric strings: dot-decimal (e.g. 1234.56) or Brazilian (1.234,56).
    """
    s_str = (
        s.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "NULL": pd.NA,
            "null": pd.NA,
            "-": pd.NA,
            "...": pd.NA,
        })
    )
    normalized = s_str.copy()
    br_mask = normalized.str.contains(",", na=False)
    normalized.loc[br_mask] = (
        normalized.loc[br_mask]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(normalized, errors="coerce")


## RAIS 2022

In [ ]:
def ler_arquivo_rais_municipio(path_arquivo: Path) -> pd.DataFrame:
    """
    Lê arquivo RAIS agregado por município.

    Suporta:
    - .csv
    - .csv.gz
    - .parquet
    - .xlsx
    - .xls
    """
    path_arquivo = Path(path_arquivo)

    if not path_arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path_arquivo}")

    suffixes = "".join(path_arquivo.suffixes).lower()

    if suffixes.endswith(".parquet"):
        df = pd.read_parquet(path_arquivo)

    elif suffixes.endswith(".csv.gz"):
        df = pd.read_csv(path_arquivo, dtype="string", compression="gzip")

    elif suffixes.endswith(".csv"):
        try:
            df = pd.read_csv(path_arquivo, dtype="string", encoding="utf-8")

            if df.shape[1] == 1:
                df = pd.read_csv(path_arquivo, sep=";", dtype="string", encoding="utf-8")

        except UnicodeDecodeError:
            df = pd.read_csv(path_arquivo, dtype="string", encoding="latin1")

            if df.shape[1] == 1:
                df = pd.read_csv(path_arquivo, sep=";", dtype="string", encoding="latin1")

    elif suffixes.endswith(".xlsx") or suffixes.endswith(".xls"):
        df = pd.read_excel(path_arquivo, dtype="string")

    else:
        raise ValueError(f"Extensão não suportada: {suffixes}")

    df = normalizar_colunas(df)

    return df


In [ ]:
df_rais_raw = ler_arquivo_rais_municipio(path_rais)

print(df_rais_raw.shape)
df_rais_raw.head()


In [ ]:
df_rais_raw.dtypes


## Padronização

In [ ]:
def padronizar_rais_2022_municipio(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza a RAIS 2022 agregada por município.

    Saída:
        codigo_ibge
        ano
        variáveis agregadas da RAIS
    """
    df = normalizar_colunas(df_raw)

    colunas_obrigatorias = [
        "ano",
        "sigla_uf",
        "id_municipio",
        "nome_municipio",
        "qtd_vinculos_rais",
        "qtd_vinculos_ativos_3112",
        "remuneracao_media",
        "massa_remuneracao_media",
        "qtd_vinculos_ativos_comercio_servicos",
        "qtd_vinculos_ativos_industria_construcao",
        "qtd_vinculos_ativos_agro",
    ]

    ausentes = [col for col in colunas_obrigatorias if col not in df.columns]

    if ausentes:
        raise ValueError(
            f"As seguintes colunas obrigatórias não foram encontradas: {ausentes}. "
            f"Colunas disponíveis: {df.columns.tolist()}"
        )

    out = pd.DataFrame()

    out["codigo_ibge"] = (
        df["id_municipio"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")

    out["sigla_uf_origem"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    out["nome_municipio_origem"] = df["nome_municipio"].astype("string").str.strip()

    cols_numericas = [
        "qtd_vinculos_rais",
        "qtd_vinculos_ativos_3112",
        "remuneracao_media",
        "massa_remuneracao_media",
        "qtd_vinculos_ativos_comercio_servicos",
        "qtd_vinculos_ativos_industria_construcao",
        "qtd_vinculos_ativos_agro",
    ]

    for col in cols_numericas:
        out[col] = parse_numero(df[col])

    # Garante ano 2022
    out = out[out["ano"] == 2022].copy()

    # Remove duplicidades, caso existam
    duplicadas = out.duplicated(["codigo_ibge", "ano"]).sum()

    if duplicadas > 0:
        print(f"Atenção: existem {duplicadas} duplicidades por codigo_ibge-ano. Consolidando.")

        out = (
            out
            .groupby(["codigo_ibge", "ano"], as_index=False)
            .agg(
                sigla_uf_origem=("sigla_uf_origem", "first"),
                nome_municipio_origem=("nome_municipio_origem", "first"),
                qtd_vinculos_rais=("qtd_vinculos_rais", "sum"),
                qtd_vinculos_ativos_3112=("qtd_vinculos_ativos_3112", "sum"),
                remuneracao_media=("remuneracao_media", "mean"),
                massa_remuneracao_media=("massa_remuneracao_media", "sum"),
                qtd_vinculos_ativos_comercio_servicos=("qtd_vinculos_ativos_comercio_servicos", "sum"),
                qtd_vinculos_ativos_industria_construcao=("qtd_vinculos_ativos_industria_construcao", "sum"),
                qtd_vinculos_ativos_agro=("qtd_vinculos_ativos_agro", "sum"),
            )
        )

    # Shares setoriais sobre vínculos ativos
    denom = out["qtd_vinculos_ativos_3112"].replace(0, np.nan)

    out["share_vinculos_comercio_servicos"] = (
        out["qtd_vinculos_ativos_comercio_servicos"] / denom
    )

    out["share_vinculos_industria_construcao"] = (
        out["qtd_vinculos_ativos_industria_construcao"] / denom
    )

    out["share_vinculos_agro"] = (
        out["qtd_vinculos_ativos_agro"] / denom
    )
    out = (
        out
        .sort_values(["codigo_ibge", "ano"])
        .reset_index(drop=True)
    )

    return out


In [ ]:
rais_2022_municipio = padronizar_rais_2022_municipio(df_rais_raw)

print(rais_2022_municipio.shape)
rais_2022_municipio.head()


In [ ]:
cols_rais = [
    "codigo_ibge",
    "ano",
    "sigla_uf_origem",
    "nome_municipio_origem",
    "qtd_vinculos_rais",
    "qtd_vinculos_ativos_3112",
    "remuneracao_media",
    "massa_remuneracao_media",
    "qtd_vinculos_ativos_comercio_servicos",
    "qtd_vinculos_ativos_industria_construcao",
    "qtd_vinculos_ativos_agro",
    "share_vinculos_comercio_servicos",
    "share_vinculos_industria_construcao",
    "share_vinculos_agro",
]


In [ ]:
rais_2022_municipio[cols_rais]


In [ ]:
rais_2022_municipio[cols_rais].to_parquet(
    TRUSTED_IBGE_DIR / "rais_2022_municipio.parquet",
    index=False
)
